In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

TABLA_BRONZE = "proyecto_final.bronze.resenas"
TABLA_SILVER = "proyecto_final.silver.resenas"
TABLA_CUARENTENA = "proyecto_final.silver.cuarentena"
TABLA_CATALOGO = "proyecto_final.silver.catalogo"

df_bronze = spark.table(TABLA_BRONZE)

print("=== ESQUEMA BRONZE RESEÑAS ===")
df_bronze.printSchema()

print("\n=== TOTAL DE REGISTROS ===")
print(df_bronze.count())

print("\n=== MUESTRA DE RESEÑAS ===")
display(df_bronze.limit(10))

In [0]:
df_resumen = df_bronze.agg(
    F.count("*").alias("total_registros"),

    F.countDistinct("resena_id")
        .alias("resenas_distintas"),

    F.sum(
        (
            F.col("resena_id").isNull() |
            (F.trim(F.col("resena_id")) == "")
        ).cast("int")
    ).alias("resena_id_vacio"),

    F.sum(
        (
            F.col("producto_id").isNull() |
            (F.trim(F.col("producto_id")) == "")
        ).cast("int")
    ).alias("producto_id_vacio"),

    F.sum(
        (
            F.col("cliente_id").isNull() |
            (F.trim(F.col("cliente_id")) == "")
        ).cast("int")
    ).alias("cliente_id_vacio"),

    F.sum(
        F.col("calificacion").isNull().cast("int")
    ).alias("calificacion_nula"),

    F.sum(
        F.col("comentario").isNull().cast("int")
    ).alias("comentario_nulo"),

    F.sum(
        F.col("tags").isNull().cast("int")
    ).alias("tags_nulo")
)

print("=== RESUMEN DE CALIDAD BRONZE ===")
display(df_resumen)

In [0]:
from pyspark.sql import functions as F

print("=== DISTRIBUCIÓN POR ARCHIVO ===")

display(
    df_bronze
    .groupBy("archivo_origen")
    .agg(
        F.count("*").alias("total_registros"),
        F.countDistinct("resena_id").alias("resenas_distintas"),
        F.min("fecha_ingesta").alias("primera_ingesta"),
        F.max("fecha_ingesta").alias("ultima_ingesta")
    )
    .orderBy("archivo_origen")
)

print("=== VALIDACIÓN DE CAMPOS ANIDADOS ===")

display(
    df_bronze.agg(
        F.count("*").alias("total_registros"),
        F.sum(F.col("tags").isNull().cast("int"))
            .alias("tags_nulos"),
        F.sum(F.col("respuestas").isNull().cast("int"))
            .alias("respuestas_nulas"),
        F.sum(F.col("metadata").isNull().cast("int"))
            .alias("metadata_nula"),
        F.sum(
            (F.size("respuestas") > 0).cast("int")
        ).alias("resenas_con_respuestas")
    )
)

print("=== MUESTRA DE RESPUESTAS Y METADATA ===")

display(
    df_bronze
    .filter(F.size("respuestas") > 0)
    .select(
        "resena_id",
        "respuestas",
        "metadata",
        "tags"
    )
    .limit(10)
)

In [0]:
from pyspark.sql import functions as F

# Estructura definida en resenas.yml
schema_respuesta = (
    "STRUCT<autor:STRING,texto:STRING,fecha:STRING>"
)

# Extraer las respuestas individuales de cada reseña
df_respuestas_explotadas = (
    df_bronze
    .select(
        "resena_id",
        F.explode_outer("respuestas").alias("respuesta_json")
    )
    .filter(F.col("respuesta_json").isNotNull())
    .withColumn(
        "respuesta_struct",
        F.from_json(
            F.col("respuesta_json"),
            schema_respuesta
        )
    )
)

print("=== VALIDACIÓN DE RESPUESTAS JSON ===")

display(
    df_respuestas_explotadas.agg(
        F.count("*").alias("total_respuestas"),

        F.sum(
            F.col("respuesta_struct").isNull().cast("int")
        ).alias("json_no_convertible"),

        F.sum(
            (
                F.col("respuesta_struct.autor").isNull() |
                (F.trim(F.col("respuesta_struct.autor")) == "")
            ).cast("int")
        ).alias("respuestas_sin_autor"),

        F.sum(
            F.col("respuesta_struct.texto").isNull().cast("int")
        ).alias("respuestas_sin_texto"),

        F.sum(
            F.col("respuesta_struct.fecha").isNull().cast("int")
        ).alias("respuestas_sin_fecha")
    )
)

print("=== MUESTRA DE RESPUESTAS CONVERTIDAS ===")

display(
    df_respuestas_explotadas
    .select(
        "resena_id",
        "respuesta_struct"
    )
    .limit(10)
)

In [0]:
from pyspark.sql import functions as F

df_resenas_sin_autor = (
    df_respuestas_explotadas
    .filter(
        F.col("respuesta_struct.autor").isNull()
        | (F.trim(F.col("respuesta_struct.autor")) == "")
    )
    .select("resena_id")
    .distinct()
)

print("=== RESEÑAS CON RESPUESTAS SIN AUTOR ===")

display(
    df_resenas_sin_autor.agg(
        F.count("*").alias("resenas_afectadas")
    )
)

print("=== IDENTIFICADORES PARA CUARENTENA ===")

display(
    df_resenas_sin_autor.orderBy("resena_id")
)

In [0]:
from pyspark.sql import functions as F

df_validacion = (
    df_bronze
    .withColumn(
        "calificacion_int",
        F.col("calificacion").cast("int")
    )
    .withColumn(
        "fecha_resena_date",
        F.coalesce(
            F.try_to_timestamp(
                "fecha_resena",
                F.lit("yyyy-MM-dd")
            ).cast("date"),
            F.try_to_timestamp(
                "fecha_resena",
                F.lit("dd/MM/yyyy")
            ).cast("date")
        )
    )
    .withColumn(
        "calificacion_invalida",
        F.col("calificacion_int").isNull() |
        (~F.col("calificacion_int").between(1, 5))
    )
    .withColumn(
        "fecha_invalida",
        F.col("fecha_resena_date").isNull()
    )
)

print("=== DISTRIBUCIÓN DE CALIFICACIONES ===")

display(
    df_validacion
    .groupBy("calificacion")
    .count()
    .orderBy("calificacion")
)

print("=== RESUMEN DE CALIDAD ===")

display(
    df_validacion.agg(
        F.count("*").alias("total_bronze"),

        F.sum(
            F.col("calificacion_invalida").cast("int")
        ).alias("calificaciones_invalidas"),

        F.sum(
            F.col("fecha_invalida").cast("int")
        ).alias("fechas_invalidas"),

        F.min("fecha_resena_date").alias("fecha_minima"),

        F.max("fecha_resena_date").alias("fecha_maxima"),

        F.sum(
            F.col("calificacion_int").isin(1, 2).cast("int")
        ).alias("resenas_negativas")
    )
)

In [0]:
from pyspark.sql import functions as F

df_catalogo_silver = (
    spark.table("proyecto_final.silver.catalogo")
    .select("producto_id")
    .distinct()
    .withColumn("existe_silver", F.lit(True))
)

df_catalogo_bronze = (
    spark.table("proyecto_final.bronze.catalogo")
    .select("producto_id")
    .distinct()
    .withColumn("existe_bronze", F.lit(True))
)

df_referencias = (
    df_validacion
    .join(df_catalogo_silver, "producto_id", "left")
    .join(df_catalogo_bronze, "producto_id", "left")
    .withColumn(
        "estado_producto",
        F.when(
            F.col("existe_silver") == True,
            "EXISTE_EN_SILVER"
        ).when(
            F.col("existe_bronze") == True,
            "SOLO_EN_BRONZE"
        ).otherwise("HUERFANO")
    )
)

print("=== VALIDACION DE REFERENCIAS AL CATALOGO ===")

display(
    df_referencias
    .groupBy("estado_producto")
    .agg(
        F.count("*").alias("total_resenas"),
        F.countDistinct("producto_id").alias("productos_distintos")
    )
    .orderBy("estado_producto")
)

In [0]:
from pyspark.sql import functions as F

df_solo_bronze = (
    df_referencias
    .filter(F.col("estado_producto") == "SOLO_EN_BRONZE")
    .select("producto_id")
    .distinct()
)

df_catalogo_revision = (
    spark.table("proyecto_final.bronze.catalogo")
    .join(df_solo_bronze, "producto_id", "inner")
)

print("=== ESTRUCTURA DE LOS PRODUCTOS SOLO EN BRONZE ===")
df_catalogo_revision.printSchema()

print("=== RESUMEN DE LOS 48 PRODUCTOS ===")

display(
    df_catalogo_revision.agg(
        F.countDistinct("producto_id").alias("productos_distintos"),
        F.count("*").alias("registros_catalogo"),
        F.sum(
            (
                F.col("categoria").isNull()
                | (F.trim(F.col("categoria")) == "")
            ).cast("int")
        ).alias("categorias_vacias")
    )
)

print("=== CATEGORÍAS ENCONTRADAS ===")

display(
    df_catalogo_revision
    .groupBy("categoria")
    .agg(
        F.countDistinct("producto_id").alias("productos_distintos")
    )
    .orderBy("categoria")
)

In [0]:
from pyspark.sql import functions as F

# Contar identificadores repetidos
df_duplicados = (
    df_bronze
    .groupBy("resena_id")
    .agg(
        F.count("*").alias("total_apariciones"),
        F.countDistinct("archivo_origen").alias("archivos_distintos"),
        F.countDistinct(
            F.sha2(
                F.to_json(
                    F.struct(
                        "producto_id",
                        "cliente_id",
                        "calificacion",
                        "comentario",
                        "fecha_resena",
                        "tags",
                        "respuestas",
                        "metadata"
                    )
                ),
                256
            )
        ).alias("versiones_contenido")
    )
    .filter(F.col("total_apariciones") > 1)
)

print("=== RESUMEN DE DUPLICADOS ===")

display(
    df_duplicados.agg(
        F.count("*").alias("resenas_repetidas"),
        F.sum(
            F.col("total_apariciones") - 1
        ).alias("filas_adicionales"),
        F.sum(
            (F.col("versiones_contenido") > 1).cast("int")
        ).alias("resenas_con_cambios"),
        F.sum(
            (F.col("versiones_contenido") == 1).cast("int")
        ).alias("resenas_identicas"),
        F.sum(
            (F.col("archivos_distintos") > 1).cast("int")
        ).alias("repetidas_entre_lotes")
    )
)

print("=== MUESTRA DE DUPLICADOS ===")

display(
    df_duplicados
    .orderBy(F.desc("versiones_contenido"), "resena_id")
    .limit(15)
)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# 1. Deduplicar: conservar una aparición por reseña
w_resena = (
    Window.partitionBy("resena_id")
    .orderBy(
        F.col("fecha_ingesta").desc(),
        F.col("archivo_origen").desc()
    )
)

df_resenas_unicas = (
    df_referencias
    .withColumn("orden_duplicado", F.row_number().over(w_resena))
    .filter(F.col("orden_duplicado") == 1)
    .drop("orden_duplicado")
)

# 2. Identificar reseñas con respuestas sin autor
df_autores_invalidos = (
    df_resenas_sin_autor
    .withColumn("respuesta_sin_autor", F.lit(True))
)

# 3. Consolidar las reglas críticas
df_calidad_resenas = (
    df_resenas_unicas
    .join(df_autores_invalidos, "resena_id", "left")
    .withColumn(
        "rechazo_calificacion",
        F.col("calificacion_invalida")
    )
    .withColumn(
        "rechazo_fecha",
        F.col("fecha_invalida")
    )
    .withColumn(
        "rechazo_producto",
        F.col("estado_producto") == "HUERFANO"
    )
    .withColumn(
        "rechazo_autor",
        F.coalesce(F.col("respuesta_sin_autor"), F.lit(False))
    )
    .withColumn(
        "rechazado",
        F.col("rechazo_calificacion") |
        F.col("rechazo_fecha") |
        F.col("rechazo_producto") |
        F.col("rechazo_autor")
    )
    .withColumn(
        "motivo_rechazo",
        F.concat_ws(
            ", ",
            F.when(F.col("rechazo_calificacion"), "CALIFICACION_INVALIDA"),
            F.when(F.col("rechazo_fecha"), "FECHA_INVALIDA"),
            F.when(F.col("rechazo_producto"), "PRODUCTO_HUERFANO"),
            F.when(F.col("rechazo_autor"), "RESPUESTA_SIN_AUTOR")
        )
    )
)

print("=== CONCILIACION DE CALIDAD RESEÑAS ===")

display(
    df_calidad_resenas.agg(
        F.count("*").alias("resenas_unicas"),
        F.sum(F.col("rechazo_calificacion").cast("int"))
            .alias("calificaciones_invalidas"),
        F.sum(F.col("rechazo_fecha").cast("int"))
            .alias("fechas_invalidas"),
        F.sum(F.col("rechazo_producto").cast("int"))
            .alias("productos_huerfanos"),
        F.sum(F.col("rechazo_autor").cast("int"))
            .alias("respuestas_sin_autor"),
        F.sum(F.col("rechazado").cast("int"))
            .alias("total_rechazadas"),
        F.sum((~F.col("rechazado")).cast("int"))
            .alias("total_aceptadas")
    )
)

print("=== REFERENCIAS DE PRODUCTOS ACEPTADOS ===")

display(
    df_calidad_resenas
    .filter(~F.col("rechazado"))
    .groupBy("estado_producto")
    .count()
    .orderBy("estado_producto")
)

In [0]:
from pyspark.sql import functions as F

df_silver_resenas = (
    df_calidad_resenas
    .filter(~F.col("rechazado"))
    .select(
        F.trim("resena_id").alias("resena_id"),
        F.trim("producto_id").alias("producto_id"),
        F.trim("cliente_id").alias("cliente_id"),

        F.col("calificacion_int")
            .cast("int")
            .alias("calificacion"),

        F.col("comentario")
            .cast("string")
            .alias("comentario"),

        F.coalesce(
            F.col("tags"),
            F.array().cast("array<string>")
        ).alias("tags"),

        F.transform(
            F.col("respuestas"),
            lambda r: F.from_json(
                r,
                "STRUCT<autor:STRING,texto:STRING,fecha:STRING>"
            )
        ).alias("respuestas"),

        F.col("fecha_resena_date")
            .alias("fecha_resena"),

        F.to_json("metadata").alias("metadata")
    )
)

print("=== VALIDACIÓN PREVIA A SILVER ===")

display(
    df_silver_resenas.agg(
        F.count("*").alias("total_resenas"),
        F.countDistinct("resena_id").alias("resenas_distintas"),
        F.sum(F.col("tags").isNull().cast("int"))
            .alias("tags_nulos"),
        F.sum(F.col("respuestas").isNull().cast("int"))
            .alias("respuestas_nulas"),
        F.sum(F.col("metadata").isNull().cast("int"))
            .alias("metadata_nula")
    )
)

print("=== ESQUEMA FINAL SILVER ===")
df_silver_resenas.printSchema()

In [0]:
TABLA_SILVER = "proyecto_final.silver.resenas"

(
    df_silver_resenas.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABLA_SILVER)
)

print("Tabla Silver Reseñas creada correctamente.")

In [0]:
%sql
SELECT
    COUNT(*) AS total_resenas,
    COUNT(DISTINCT resena_id) AS resenas_distintas,
    SUM(CASE WHEN calificacion IN (1,2) THEN 1 ELSE 0 END)
        AS resenas_negativas,
    SUM(CASE WHEN SIZE(tags) = 0 THEN 1 ELSE 0 END)
        AS tags_vacios,
    SUM(CASE WHEN SIZE(respuestas) > 0 THEN 1 ELSE 0 END)
        AS resenas_con_respuestas,
    MIN(fecha_resena) AS fecha_minima,
    MAX(fecha_resena) AS fecha_maxima
FROM proyecto_final.silver.resenas;

In [0]:
%sql
SELECT
    resena_id,
    producto_id,
    cliente_id,
    calificacion,
    comentario,
    tags,
    respuestas,
    fecha_resena,
    metadata
FROM proyecto_final.silver.resenas
ORDER BY resena_id
LIMIT 30;

In [0]:
%sql
SELECT
    resena_id,
    calificacion,
    comentario,
    respuestas[0].autor AS respuesta_autor,
    respuestas[0].texto AS respuesta_texto,
    respuestas[0].fecha AS respuesta_fecha,
    tags,
    metadata
FROM proyecto_final.silver.resenas
WHERE SIZE(respuestas) > 0
ORDER BY resena_id
LIMIT 15;

In [0]:
from pyspark.sql import functions as F

# Conservar las columnas originales de Bronze
columnas_bronze = df_bronze.columns

df_cuarentena_resenas = (
    df_calidad_resenas
    .filter(F.col("rechazado"))
    .select(
        F.lit("resenas").alias("fuente"),
        F.col("resena_id").alias("clave_registro"),
        F.col("motivo_rechazo"),
        F.current_timestamp().alias("fecha_rechazo"),
        F.to_json(
            F.struct(*[
                F.col(c) for c in columnas_bronze
            ])
        ).alias("datos_originales"),
        F.col("archivo_origen")
    )
)

print("=== REGISTROS PREPARADOS PARA CUARENTENA ===")

display(
    df_cuarentena_resenas
    .groupBy("motivo_rechazo")
    .count()
    .orderBy("motivo_rechazo")
)

assert df_cuarentena_resenas.count() == 223, \
    "La cuarentena no coincide con los 223 rechazos."

print("Total de rechazos validado: 223")

In [0]:
(
    df_cuarentena_resenas.write
    .format("delta")
    .mode("overwrite")
    .option("replaceWhere", "fuente = 'resenas'")
    .saveAsTable("proyecto_final.silver.cuarentena")
)

print("Cuarentena de Reseñas cargada correctamente.")

In [0]:
%sql
SELECT
    fuente,
    motivo_rechazo,
    COUNT(*) AS total
FROM proyecto_final.silver.cuarentena
WHERE fuente = 'resenas'
GROUP BY fuente, motivo_rechazo
ORDER BY total DESC;

In [0]:
%sql
SELECT
*
FROM proyecto_final.silver.cuarentena
WHERE fuente = 'resenas'
